# !Advertencia!
Este notebook es solo para realizar pruebas no es el Notebook Oficial del Modelo

# Pruebas con Trashnet

In [5]:
import os
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, as_completed

def process_single_image(src_path, dest_path):
    """
    Aplica la transformación de 224x224 (Letterboxing) a una sola imagen
    y la guarda en la ruta de destino.
    """
    try:
        with Image.open(src_path) as img:
            # 1. Forzar a RGB para evitar problemas de canales alfa
            if img.mode in ("RGBA", "P"):
                img = img.convert("RGB")
                
            # 2. Transformación a 224x224 manteniendo la proporción
            TARGET_SIZE = (224, 224)
            img.thumbnail(TARGET_SIZE, Image.Resampling.LANCZOS)
            
            # 3. Crear lienzo negro y centrar
            final_img = Image.new("RGB", TARGET_SIZE, (0, 0, 0))
            paste_x = (TARGET_SIZE[0] - img.width) // 2
            paste_y = (TARGET_SIZE[1] - img.height) // 2
            
            final_img.paste(img, (paste_x, paste_y))
            
            # 4. Guardar
            final_img.save(dest_path, format="JPEG", quality=95)
            return True
            
    except Exception as e:
        print(f"❌ Error procesando la imagen {src_path}: {str(e)}")
        return False


def format_dataset_for_mobilenet(input_dir, output_dir, max_workers=8):
    """
    Lee un directorio con subcarpetas de clases, estandariza las imágenes a 224x224 
    (MobileNetV3 format) y las guarda en un nuevo directorio manteniendo la estructura.
    """
    classes = ["glass", "paper", "cardboard", "plastic", "metal", "trash"]
    
    # Validar que el directorio de entrada exista
    if not os.path.exists(input_dir):
        print(f"🚨 ERROR: El directorio de origen '{input_dir}' no existe.")
        return
        
    print(f"🚀 Iniciando estandarización para MobileNetV3 (224x224)...")
    print(f"📂 Origen: {os.path.abspath(input_dir)}")
    print(f"📂 Destino: {os.path.abspath(output_dir)}")
    print("-" * 50)

    # Recolectar todas las tareas (imágenes a procesar)
    tasks = []
    
    for cls in classes:
        cls_input_dir = os.path.join(input_dir, cls)
        cls_output_dir = os.path.join(output_dir, cls)
        
        # Si la carpeta de clase no existe en el origen, nos la saltamos
        if not os.path.exists(cls_input_dir):
            print(f"⚠️ Alerta: No se encontró la carpeta '{cls}' en el origen.")
            continue
            
        # Crear la carpeta equivalente en el destino
        os.makedirs(cls_output_dir, exist_ok=True)
        
        # Leer todas las imágenes de esta carpeta
        for filename in os.listdir(cls_input_dir):
            # Filtro básico para ignorar archivos de sistema como .DS_Store
            if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.webp')):
                src_path = os.path.join(cls_input_dir, filename)
                # Forzamos la extensión a .jpg en el destino
                name_without_ext = os.path.splitext(filename)[0]
                dest_path = os.path.join(cls_output_dir, f"{name_without_ext}.jpg")
                
                tasks.append((src_path, dest_path))

    total_images = len(tasks)
    if total_images == 0:
        print("⚠️ No se encontraron imágenes válidas en el directorio de origen.")
        return

    print(f"Encontradas {total_images} imágenes. Procesando en paralelo con {max_workers} hilos...")
    
    # Procesar en paralelo
    success_count = 0
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Enviar todas las tareas al pool
        futures = {executor.submit(process_single_image, src, dest): src for src, dest in tasks}
        
        # Procesar resultados conforme van terminando
        for future in as_completed(futures):
            if future.result():
                success_count += 1

    print("-" * 50)
    print(f"✅ Proceso completado.")
    print(f"📊 Imágenes procesadas exitosamente: {success_count} / {total_images}")

In [7]:
taco_dir = "/Users/jairpalma/Development/Reciclame/dataset-resized"
output_dir = "./dataset/TrashNet_clipped"
format_dataset_for_mobilenet(input_dir=taco_dir, output_dir=output_dir, max_workers=10)

🚀 Iniciando estandarización para MobileNetV3 (224x224)...
📂 Origen: /Users/jairpalma/Development/Reciclame/dataset-resized
📂 Destino: /Users/jairpalma/Development/Reciclame/vision_model/notebooks/dataset/TrashNet_clipped
--------------------------------------------------
Encontradas 2521 imágenes. Procesando en paralelo con 10 hilos...
--------------------------------------------------
✅ Proceso completado.
📊 Imágenes procesadas exitosamente: 2521 / 2521
